In [40]:
import pandas as pd

# Carregar os dois arquivos
df_train = pd.read_csv("../data/processed/df_train_preprocessed.csv", index_col=0)

# Criar tabela de contagem por categoria
tabela_categorias = df_train["condition_label"].value_counts().reset_index()
tabela_categorias.columns = ["Categoria", "Quantidade"]

print(tabela_categorias)


   Categoria  Quantidade
0          5        3844
1          1        2530
2          4        2441
3          3        1540
4          2        1195


In [41]:
df_train.info()

<class 'pandas.DataFrame'>
RangeIndex: 11550 entries, 0 to 11549
Data columns (total 3 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   condition_label   11550 non-null  int64
 1   medical_abstract  11550 non-null  str  
 2   condition_name    11550 non-null  str  
dtypes: int64(1), str(2)
memory usage: 270.8 KB


In [39]:
df_train['medical_abstract'] = df_train['medical_abstract'].str.lower()
df_train['medical_abstract'] = df_train['medical_abstract'].str.replace(r'\[.*?\]','', regex=True)
df_train['medical_abstract'] = df_train['medical_abstract'].str.replace(r'[^\w\s]','', regex=True)

display(df_train)

,Unnamed: 0,condition_label,medical_abstract,condition_name
0,0,5,tissue changes around loose prostheses a canin...,general pathological conditions
1,1,1,neuropeptide y and neuronspecific enolase leve...,neoplasms
2,2,2,sexually transmitted diseases of the colon rec...,digestive system diseases
3,3,1,lipolytic factors associated with murine and h...,neoplasms
4,4,3,does carotid restenosis predict an increased r...,nervous system diseases
...,...,...,...,...
11545,11545,1,epirubicin at two dose levels with prednisolon...,neoplasms
11546,11546,1,four and a half year follow up of women with d...,neoplasms
11547,11547,5,safety of the transbronchial biopsy in outpati...,general pathological conditions
11548,11548,3,interictal spikes and hippocampal somatostatin...,nervous system diseases


In [42]:
def lower_replace(series):
    output = series.str.lower()
    output = output.str.replace(r'\[.*?\]','', regex=True)
    output = output.str.replace(r'[^\w\s]','', regex=True)

    return output

In [43]:
lower_replace(df_train.medical_abstract)

0        tissue changes around loose prostheses a canin...
1        neuropeptide y and neuronspecific enolase leve...
2        sexually transmitted diseases of the colon rec...
3        lipolytic factors associated with murine and h...
4        does carotid restenosis predict an increased r...
                               ...                        
11545    epirubicin at two dose levels with prednisolon...
11546    four and a half year follow up of women with d...
11547    safety of the transbronchial biopsy in outpati...
11548    interictal spikes and hippocampal somatostatin...
11549    recurrent thoracic outlet syndrome after first...
Name: medical_abstract, Length: 11550, dtype: str

In [44]:
df = df_train.apply(lambda col: lower_replace(col) if col.dtype == "str" else col)
df.head(10)

,condition_label,medical_abstract,condition_name
0,5,tissue changes around loose prostheses a canin...,general pathological conditions
1,1,neuropeptide y and neuronspecific enolase leve...,neoplasms
2,2,sexually transmitted diseases of the colon rec...,digestive system diseases
3,1,lipolytic factors associated with murine and h...,neoplasms
4,3,does carotid restenosis predict an increased r...,nervous system diseases
5,3,the shoulder in multiple epiphyseal dysplasia ...,nervous system diseases
6,2,the management of postoperative chylous ascite...,digestive system diseases
7,4,pharmacomechanical thrombolysis and angioplast...,cardiovascular diseases
8,5,color doppler diagnosis of mechanical prosthet...,general pathological conditions
9,5,noninvasive diagnosis of rightsided extracardi...,general pathological conditions


In [45]:
df.to_csv('../data/processed/data_process_pandas_1.csv')

In [46]:
df_process = pd.read_csv('../data/processed/data_process_pandas_1.csv', index_col=0)
df_process

,condition_label,medical_abstract,condition_name
0,5,tissue changes around loose prostheses a canin...,general pathological conditions
1,1,neuropeptide y and neuronspecific enolase leve...,neoplasms
2,2,sexually transmitted diseases of the colon rec...,digestive system diseases
3,1,lipolytic factors associated with murine and h...,neoplasms
4,3,does carotid restenosis predict an increased r...,nervous system diseases
...,...,...,...
11545,1,epirubicin at two dose levels with prednisolon...,neoplasms
11546,1,four and a half year follow up of women with d...,neoplasms
11547,5,safety of the transbronchial biopsy in outpati...,general pathological conditions
11548,3,interictal spikes and hippocampal somatostatin...,nervous system diseases


In [ ]:
import re
import spacy

nlp = spacy.load("en_core_web_sm")

# Função para normalizar texto
def lower_replace(text: str) -> str:
    text = text.lower()
    text = re.sub(r'\[.*?\]', '', text)       # remove conteúdo entre colchetes
    text = re.sub(r'[^\w\s]', '', text)       # remove pontuação
    return text

# Tokenização + lematização + remoção de stopwords
def token_lemma_stop(text: str) -> list:
    doc = nlp(text)
    return [token.lemma_ for token in doc if not token.is_stop]

# Filtrar apenas certas classes gramaticais (exemplo: substantivos e adjetivos)
def filter_pos(tokens: list) -> str:
    doc = nlp(" ".join(tokens))
    return " ".join([token.text for token in doc if token.pos_ in ["NOUN", "ADJ", "PRON", "VERB"]])


# Pipeline único
def preprocess(text: str) -> list:
    text = lower_replace(text)
    tokens = token_lemma_stop(text)
    return filter_pos(tokens)

# Aplicar no DataFrame
df_process['medical_abstract_clean'] = df_process['medical_abstract'].apply(preprocess)


In [49]:
df_process.head(100)

,condition_label,medical_abstract,condition_name
0,5,tissue changes around loose prostheses a canin...,general pathological conditions
1,1,neuropeptide y and neuronspecific enolase leve...,neoplasms
2,2,sexually transmitted diseases of the colon rec...,digestive system diseases
3,1,lipolytic factors associated with murine and h...,neoplasms
4,3,does carotid restenosis predict an increased r...,nervous system diseases
...,...,...,...
95,4,mucoid vasculopathy of unknown etiology a new ...,cardiovascular diseases
96,1,the importance of cytogenetic studies in adult...,neoplasms
97,5,smooth muscle cell proliferation and restenosi...,general pathological conditions
98,4,late effects of treatment for wilms tumor a re...,cardiovascular diseases


In [50]:
pd.to_pickle(df_process, '../data/processed/text_clean.pkl')

In [51]:
text_clean = pd.read_pickle('../data/processed/text_clean.pkl')
text_clean

,condition_label,medical_abstract,condition_name
0,5,tissue changes around loose prostheses a canin...,general pathological conditions
1,1,neuropeptide y and neuronspecific enolase leve...,neoplasms
2,2,sexually transmitted diseases of the colon rec...,digestive system diseases
3,1,lipolytic factors associated with murine and h...,neoplasms
4,3,does carotid restenosis predict an increased r...,nervous system diseases
...,...,...,...
11545,1,epirubicin at two dose levels with prednisolon...,neoplasms
11546,1,four and a half year follow up of women with d...,neoplasms
11547,5,safety of the transbronchial biopsy in outpati...,general pathological conditions
11548,3,interictal spikes and hippocampal somatostatin...,nervous system diseases


In [52]:
# # Count Vectorizer

# text_clean = pd.read_pickle('../data/processed/text_clean.pkl')

# from sklearn.feature_extraction.text import CountVectorizer

# cv2 = CountVectorizer(stop_words='english', ngram_range=(1,2), min_df=0.1, max_df=0.8)
# dtm2 = cv2.fit_transform(text_clean['medical_abstract_clean'])

# dtm2_df = pd.DataFrame(dtm2.toarray(), columns=cv2.get_feature_names_out())
# dtm2_df

In [53]:
# term_freq = dtm2_df.sum()
# term_freq = term_freq.head(10)

# term_freq.sort_values().plot(kind='barh');

In [58]:
# modelo de baseline TF_IDF
from sklearn.feature_extraction.text import TfidfVectorizer

tv2 = TfidfVectorizer(stop_words='english', ngram_range=(1,2), min_df=0.2, max_df=0.8)
tfidf2 = tv2.fit_transform(text_clean.medical_abstract)
tfidf_df2 = pd.DataFrame(tfidf2.toarray(), columns=tv2.get_feature_names_out())
tfidf_df2

,clinical,disease,patient,patients,results,study,treatment
0,0.745778,0.000000,0.000000,0.000000,0.000000,0.666194,0.000000
1,0.000000,0.426375,0.000000,0.794758,0.431931,0.000000,0.000000
2,0.000000,0.614587,0.212023,0.636435,0.415064,0.000000,0.000000
3,0.861234,0.000000,0.000000,0.508208,0.000000,0.000000,0.000000
4,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...
11545,0.000000,0.518061,0.178723,0.429182,0.174937,0.000000,0.696327
11546,0.000000,0.000000,0.878212,0.263615,0.000000,0.399062,0.000000
11547,0.000000,0.226411,0.234325,0.844055,0.000000,0.425912,0.000000
11548,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [74]:
# modelo de baseline TF_IDF
from sklearn.feature_extraction.text import TfidfVectorizer

tv2 = TfidfVectorizer(stop_words='english', ngram_range=(1,2), min_df=0.2, max_df=0.8)
tfidf2 = tv2.fit_transform(text_clean.medical_abstract)
tfidf2
tfidf_df2 = pd.DataFrame(tfidf2.toarray(), columns=tv2.get_feature_names_out())
tfidf_df2

,clinical,disease,patient,patients,results,study,treatment
0,0.745778,0.000000,0.000000,0.000000,0.000000,0.666194,0.000000
1,0.000000,0.426375,0.000000,0.794758,0.431931,0.000000,0.000000
2,0.000000,0.614587,0.212023,0.636435,0.415064,0.000000,0.000000
3,0.861234,0.000000,0.000000,0.508208,0.000000,0.000000,0.000000
4,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...
11545,0.000000,0.518061,0.178723,0.429182,0.174937,0.000000,0.696327
11546,0.000000,0.000000,0.878212,0.263615,0.000000,0.399062,0.000000
11547,0.000000,0.226411,0.234325,0.844055,0.000000,0.425912,0.000000
11548,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [ ]:
# tfidf_df2.sum().sort_values().tail(10).plot(kind='barh')

In [ ]:
# tfidf_df2.sum().sort_values().head(10).plot(kind='barh')

In [60]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
import pandas as pd


# 2. Vetorização TF-IDF
tv = TfidfVectorizer(stop_words='english', ngram_range=(1,2), min_df=0.2, max_df=0.8)
X = tv.fit_transform(text_clean.medical_abstract)

# 3. Labels (escolha a coluna alvo)
y = text_clean['condition_name']  

# 4. Separar treino e teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 5. Treinar modelo Random Forest
rf_model = RandomForestClassifier(
    n_estimators=500, 
    max_depth=50, 
    random_state=42, 
    class_weight='balanced'
)
rf_model.fit(X_train, y_train)

# 6. Fazer predições
y_pred = rf_model.predict(X_test)

# 7. Avaliar
print(classification_report(y_test, y_pred))

                                 precision    recall  f1-score   support

        cardiovascular diseases       0.23      0.20      0.21       520
      digestive system diseases       0.11      0.19      0.14       224
general pathological conditions       0.25      0.09      0.13       792
                      neoplasms       0.24      0.28      0.26       479
        nervous system diseases       0.14      0.29      0.19       295

                       accuracy                           0.19      2310
                      macro avg       0.19      0.21      0.19      2310
                   weighted avg       0.22      0.19      0.18      2310



In [65]:
# Exemplo de novo texto
novo_texto = "i have a neoplasm problem"
# Pré-processar
# novo_texto_proc = preprocess(novo_texto)

# Vetorizar com o TF-IDF já treinado
X_novo = tv.transform([novo_texto_proc])
pred = rf_model.predict(X_novo)
print("Classe prevista:", pred[0])


Classe prevista: neoplasms


In [63]:
probs = rf_model.predict_proba(X_novo)
print("Probabilidades por classe:", probs)


Probabilidades por classe: [[0.18070599 0.16238738 0.22038281 0.2396383  0.19688553]]
